# ROBER: direct emulator
(initial state, t) -> y(t), fixed rate constants. Logic lives in `src/chemistry_emulator`; this notebook only runs it.

In [ ]:
# --- Colab setup: clone (or update) the repo and install it ---------------------------------
import os, subprocess, sys
REPO_URL = "https://github.com/YOUR_USERNAME/chemistry-emulator.git"   # <- change this
REPO_DIR = "/content/chemistry-emulator"

def sh(cmd):
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout[-1500:])

if os.path.isdir(REPO_DIR):
    sh(f"cd {REPO_DIR} && git pull")
elif os.path.isdir("src/chemistry_emulator") or os.path.isdir("../src/chemistry_emulator"):
    REPO_DIR = os.path.abspath("." if os.path.isdir("src/chemistry_emulator") else "..")   # running locally
else:
    sh(f"git clone {REPO_URL} {REPO_DIR}")
sh(f"cd {REPO_DIR} && pip install -q -e .")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))      # fallback if the editable install is not seen
import importlib; importlib.invalidate_caches()
print("repo:", REPO_DIR)

In [ ]:
# Re-import edited .py files automatically (edit src/..., re-run a cell, no restart needed)
%load_ext autoreload
%autoreload 2

In [ ]:
# Optional: keep data / models on Google Drive (Colab storage is wiped when the session ends)
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/chemistry-emulator"
except ImportError:
    OUT = os.path.join(REPO_DIR, "runs")        # local: runs/ is git-ignored
os.makedirs(OUT, exist_ok=True)
print("saving to", OUT)

In [ ]:
QUICK = True      # True = tiny smoke test (seconds). Set False for the real run.
N_TRAJ, N_TIMES, EPOCHS = (20, 20, 3) if QUICK else (2000, 50, 150)
N_WORKERS = 1     # >1 uses multiprocessing (Colab free has 2 CPUs)

## 1. Generate data (cached on Drive)

In [ ]:
import numpy as np
from chemistry_emulator.systems.rober import make_direct_data

path = f"{OUT}/rober_direct_{N_TRAJ}x{N_TIMES}.npz"
if os.path.exists(path):
    d = np.load(path); X, Y, TID = d["X"], d["Y"], d["TID"]
else:
    X, Y, TID = make_direct_data(N_TRAJ, N_TIMES, seed=0, n_workers=N_WORKERS)
    np.savez(path, X=X, Y=Y, TID=TID)
print(X.shape, Y.shape)

In [ ]:
from chemistry_emulator.plotting import sketch_direct_data
fig = sketch_direct_data(X, Y, TID)

## 2. Split, normalise, train

In [ ]:
from chemistry_emulator.training import prepare, train, save_run
from chemistry_emulator.models import MLP, DeepONet
from chemistry_emulator.evaluate import evaluate_direct

data = prepare(X, Y, TID, seed=0)
models = {"mlp": MLP(n_hidden=5, n_in=4, n_out=3),
          "deeponet": DeepONet(n_hidden=5, n_out=3, n_branch_in=3)}
results = {}
for name, m in models.items():
    p = train(m, data, epochs=EPOCHS)
    results[name] = p
    save_run(f"{OUT}/rober_direct_{name}.pkl", p, data)
    evaluate_direct(m, p, data)

## 3. Pushing fixes from Colab

In [ ]:
# Pushing a fix FROM Colab (optional). Simpler: edit locally / on GitHub and `git pull` in Colab.
# Needs a GitHub token stored in Colab: key icon (Secrets) > add GH_TOKEN, allow notebook access.
# from google.colab import userdata
# tok = userdata.get("GH_TOKEN")
# sh(f"cd {REPO_DIR} && git config user.email you@example.com && git config user.name you")
# sh(f"cd {REPO_DIR} && git add -A && git commit -m 'fix' && git push https://{tok}@github.com/YOUR_USERNAME/chemistry-emulator.git HEAD")